# 01 - Fundamentos de Support Vector Regression (SVR) y el Tubo $\epsilon$-Insensible

## Curso de Machine Learning: Modelos de Regresión

Las **Máquinas de Vectores de Soporte para Regresión (Support Vector Regression - SVR)**, formuladas por Vladimir Vapnik y colaboradores en 1995, extienden los principios geométricos del margen máximo de clasificación a variables continuas.

A diferencia de la regresión clásica por mínimos cuadrados (que penaliza cualquier desviación por pequeña que sea), SVR define un **tubo de tolerancia $\epsilon$-insensible**: cualquier error inferior a $\epsilon$ tiene penalización exactamente cero.

---

### Contenidos
1. Geometría de SVR y la Pérdida $\epsilon$-Insensible
2. Formulación Matemática Primal y Vectores de Soporte
3. El Truco del Kernel: Lineal, Polinomial y RBF (Gaussiano)
4. Visualización 1D Interactiva del Tubo $\epsilon$ y los Vectores de Soporte
5. Ajuste con `sklearn.svm.SVR` en el Dataset Auto MPG
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Entorno listo.")


## 1. La Función de Pérdida $\epsilon$-Insensible

La función de pérdida de Vapnik se define como:

$$L_\epsilon(y, \hat{y}) = \max(0, |y - \hat{y}| - \epsilon)$$

- Si $|y - \hat{y}| \le \epsilon$: La predicción cae dentro del tubo; el costo es $0$.
- Si $|y - \hat{y}| > \epsilon$: Se penaliza linealmente el exceso a través de variables de holgura $\xi_i, \xi_i^*$.

### Propiedad de Esparcidad (Sparsity):
Solo las observaciones que caen **en el borde o fuera del tubo $\epsilon$** se convierten en **Vectores de Soporte**. El resto de los puntos dentro del tubo podrían removerse del conjunto de datos sin alterar en absoluto la función de predicción final.


In [ ]:
# Demostracion 1D: Visualizacion del Tubo Epsilon y Vectores de Soporte
np.random.seed(42)
X_1d = np.sort(np.random.uniform(0, 5, 50)).reshape(-1, 1)
y_1d = np.sin(X_1d).ravel() + np.random.normal(0, 0.15, 50)

epsilon = 0.20
svr_demo = SVR(kernel='rbf', C=10.0, epsilon=epsilon, gamma=0.8)
svr_demo.fit(X_1d, y_1d)

x_plot = np.linspace(0, 5, 200).reshape(-1, 1)
y_pred_plot = svr_demo.predict(x_plot)

# Vectores de soporte identificados por SVR
sv_indices = svr_demo.support_
X_sv = X_1d[sv_indices]
y_sv = y_1d[sv_indices]

plt.figure(figsize=(11, 5))
plt.scatter(X_1d, y_1d, color='#2b5c8f', label='Datos (Dentro del tubo)', zorder=2)
plt.scatter(X_sv, y_sv, s=120, facecolors='none', edgecolors='#d62728', lw=2, label=f'Vectores de Soporte (n={len(X_sv)})', zorder=3)

plt.plot(x_plot, y_pred_plot, color='#2ca02c', lw=2.5, label='Ajuste SVR f(x)')
plt.fill_between(x_plot.ravel(), y_pred_plot - epsilon, y_pred_plot + epsilon, color='green', alpha=0.15, label=f'Tubo Epsilon-Insensible (+/- {epsilon})')

plt.title('Geometria de SVR: Tubo de Tolerancia y Vectores de Soporte', fontweight='bold')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.tight_layout()
plt.show()


## 2. El Truco del Kernel (Kernel Trick)

Al sustituir el producto interno $\langle \mathbf{x}, \mathbf{x}' \rangle$ por una función de kernel $K(\mathbf{x}, \mathbf{x}')$, SVR proyecta los datos a un espacio de Hilbert de dimensión infinita sin computar explícitamente las coordenadas transformadas:

- **Kernel Lineal**: $K(\mathbf{x}, \mathbf{x}') = \mathbf{x}^T \mathbf{x}'$
- **Kernel Polinomial**: $K(\mathbf{x}, \mathbf{x}') = (\gamma \mathbf{x}^T \mathbf{x}' + r)^d$
- **Kernel RBF (Radial Basis Function)**:
  $$K(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$$


In [ ]:
# Ajuste de SVR en Auto MPG con Pipeline (Escalado obligatorio)
df = pd.read_csv("../../data/auto_mpg.csv")
features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
X = df[features]
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

pipe_svr = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=20.0, epsilon=1.5, gamma='scale'))
])

pipe_svr.fit(X_train, y_train)
y_pred_svr = pipe_svr.predict(X_test)

print(f"R2 SVR (RBF):  {r2_score(y_test, y_pred_svr):.4f}")
print(f"RMSE SVR:      {np.sqrt(mean_squared_error(y_test, y_pred_svr)):.2f} mpg")
print(f"MAE SVR:       {mean_absolute_error(y_test, y_pred_svr):.2f} mpg")
print(f"Vectores de Soporte: {len(pipe_svr.named_steps['svr'].support_)} de {len(X_train)} muestras")


## 3. Conclusiones

1. **Robustez ante Outliers**: Gracias a la pérdida lineal fuera del tubo $\epsilon$, SVR es mucho menos vulnerable a valores extremos que mínimos cuadrados (MSE cuadrático).
2. **Esparcidad**: La solución depende únicamente de los vectores de soporte situados en los márgenes o fuera de ellos.
3. **Escalado Obligatorio**: SVR requiere imperativamente que las variables estén en la misma escala numérica (`StandardScaler`).

En el siguiente cuaderno, **02_Hiperparametros_Escalado_y_Benchmark_Regresion.ipynb**, analizaremos la interacción entre $C$, $\epsilon$ y $\gamma$, y realizaremos el **Gran Benchmark Comparativo** de todos los modelos de regresión del curso.
